# Projeto Prático 2 – Redução de Dimensionalidade (Fashion-MNIST)
**Machine Learning – UBI, 2026/27 · André Fazendeiro (n.º 48191)**

Este notebook corre o projeto inteiro no Google Colab:
1. clona o repositório e instala as bibliotecas que faltam;
2. executa todos os notebooks de `notebooks/` por ordem (o resultado de cada um é guardado nesse notebook);
3. mostra as tabelas e figuras principais.

Basta fazer **Runtime → Run all**. A GPU é opcional.

In [ ]:
# 1) Clonar o repositório (só se ainda não existir) e instalar as bibliotecas que o Colab não traz
import os
if not os.path.exists("/content/PROJETO2"):
    !git clone https://github.com/Andrepf03/PROJETO2.git /content/PROJETO2
%cd /content/PROJETO2

# scikit-learn fixo na versão usada para treinar o classificador guardado em models/clf_raw.joblib
!pip install -q umap-learn scikit-image datasets scikit-learn==1.9.1

In [ ]:
# 2) Verificar o ambiente
import sklearn, torch
print("scikit-learn:", sklearn.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "não ( vai ser usada a CPU)")

## Executar todos os notebooks por ordem
Os notebooks `03` (classificador em píxeis, ~14 min) e `06` (treino dos autoencoders) são **saltados por defeito**, porque os modelos já treinados estão em `models/`. Para os treinar de novo, muda `RUN_TRAINING = True`.

In [ ]:
# 3) Executar os notebooks, um a um
import subprocess, time

RUN_TRAINING = False   # True = treina de novo o classificador em píxeis (03) e os autoencoders (06)

notebooks = [
    "00_setup_check", "01_data_exploration", "02_splits", "03_baseline_raw_pixels",
    "04_pca", "05_visualization_2d", "06_autoencoder", "07_representation_check",
    "08_reconstruction_panels", "09_classification", "10_texture_hog", "11_retrieval_module",
    "12_retrieval_evaluation", "13_retrieval_panels", "14a_color_dataset", "14b_color_retrieval",
    "15_failure_cases", "16_main_conclusion",
]

for nb in notebooks:
    if nb in ("03_baseline_raw_pixels", "06_autoencoder") and not RUN_TRAINING:
        print(f"--   {nb}: saltado (modelo já está em models/)")
        continue
    start = time.time()
    result = subprocess.run(
        ["jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace",
         "--ExecutePreprocessor.kernel_name=python3", "--ExecutePreprocessor.timeout=-1",
         f"notebooks/{nb}.ipynb"],
        capture_output=True, text=True,
    )
    status = "OK" if result.returncode == 0 else "ERRO"
    print(f"{status:4s} {nb} ({time.time() - start:.0f} s)")
    if result.returncode != 0:          # se algum falhar, mostra o erro e para
        print(result.stderr[-3000:])
        break

## Resultados principais (conjunto de teste)

In [ ]:
# 4) Tabelas principais
import pandas as pd
from IPython.display import display, Image

print("Classificação (accuracy no teste):")
display(pd.concat([pd.read_csv("results/tables/09_classification_table.csv"),
                   pd.read_csv("results/tables/10_hog_classification.csv")]).round(4))

print("Retrieval (precision@k no teste, galeria = treino):")
display(pd.read_csv("results/tables/12_retrieval_precision.csv").round(4))

print("Extensão a cores (precision@k por cor e por tipo):")
display(pd.read_csv("results/tables/14b_color_retrieval.csv").round(3))

In [ ]:
# 5) Figuras principais
figures = [
    "08_reconstruction_by_class.png",
    "09_accuracy_vs_dim.png",
    "12_precision_vs_dim.png",
    "retrieval_panels/02_Pullover.png",
    "retrieval_panels/color_Handbags.png",
    "16_classification_vs_retrieval_zoom.png",
]
for f in figures:
    print(f)
    display(Image(f"results/figures/{f}", width=650))